# 00 · Build the dataset

Reads both generations of the message log — the pre-Fall-2026 Python/peewee database and the current
platform database — reconciles them into one frame, and writes the tidy files every other notebook
reads. Run this first; nothing downstream works without it.

**What this notebook is responsible for**

* joining the same student across the two databases (via their Discord identity),
* dropping messages that appear in both because the importer already carried them across,
* excluding instructor/test accounts and anything a student has asked to have deleted,
* stating plainly how much of the current term the data actually covers.

The last point matters more than it sounds: Fall 2026 is **in progress**. Every number from this term
is a partial-term number, and the report is built to say so on the page rather than leave a reader to
work it out.

In [1]:
# Put the analysis package on the path no matter where Jupyter was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "analysis" / "bloombot_analysis").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "analysis"))

import pandas as pd

from bloombot_analysis import charts, load, metrics, privacy, report, sessions, topics
from bloombot_analysis.config import CONFIG, SURFACE_LABELS, TOPICS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("as of:", CONFIG.as_of)
print("legacy db :", CONFIG.legacy_db, "(exists)" if CONFIG.legacy_db.exists() else "(missing)")
print("current db:", CONFIG.current_db, "(exists)" if CONFIG.current_db.exists() else "(missing)")
print("output    :", CONFIG.out_dir)

as of: 2026-09-25
legacy db : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/legacy.db (exists)
current db: /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/current.db (exists)
output    : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out


In [2]:
# A clean run: stale metrics from a previous dataset are a hazard, not a cache.
metrics.reset()

legacy_df = load.load_legacy()
current_df = load.load_current()
print(f"legacy rows : {len(legacy_df):,}")
print(f"current rows: {len(current_df):,}")

legacy rows : 1,410
current rows: 1,506


## Merge and provenance

Duplicates are resolved in favour of the current database's copy: it is the one the platform keeps
writing to and the one whose ids appear in a transcript. The provenance numbers below go into the
report's methodology slide verbatim — how much came from where, and how much was dropped, is not
something a reader should have to take on trust.

In [3]:
messages_df, provenance = load.merge_messages(legacy_df, current_df)
provenance

{'legacy_rows': 1410,
 'current_rows': 1506,
 'duplicates_dropped': 1322,
 'excluded_account_rows': 10,
 'rows': 1584,
 'first_message': Timestamp('2025-09-04 22:59:00'),
 'last_message': Timestamp('2026-09-24 09:10:20')}

In [4]:
coverage = (
    messages_df.groupby(["source", "surface", "semester"])
    .size()
    .reset_index(name="messages")
    .sort_values(["semester", "surface"])
)
coverage

,source,surface,semester,messages
0,current,discord,Fall 2025,658
5,legacy,discord,Fall 2025,20
1,current,discord,Fall 2026,92
6,legacy,discord,Fall 2026,40
3,current,mcp,Fall 2026,42
4,current,web,Fall 2026,40
2,current,discord,Spring 2026,664
7,legacy,discord,Spring 2026,28


## Sessions

A session is one student, one course, one surface, split on more than
`CONFIG.session_gap_minutes` of silence. Sessions with no student message (the bot talking to
itself) are dropped — they are not usage.

In [5]:
sessions_df = sessions.session_frame(messages_df)
summary = sessions.usage_summary(sessions_df)
summary

{'sessions': 259,
 'students': 54,
 'prompts': 792,
 'courses': 4,
 'median_prompts': 3.0,
 'mean_prompts': 3.057915057915058,
 'iqr_prompts': (2.0, 4.0),
 'median_duration': 13.75,
 'iqr_duration': (5.291666666666666, 22.133333333333333)}

In [6]:
completeness = sessions.term_completeness()
print(
    f"{completeness['current_term']}: day {completeness['elapsed_days']} of "
    f"{completeness['total_days']} ({completeness['fraction_elapsed']:.0%} of the term), "
    f"as of {completeness['as_of']}"
)
print(
    f"like-for-like window in {completeness['comparison_term']}: "
    f"{CONFIG.term(CONFIG.comparison_term).start} → {completeness['comparison_window_end']}"
)
completeness

Fall 2026: day 23 of 104 (22% of the term), as of 2026-09-25
like-for-like window in Fall 2025: 2025-09-03 → 2025-09-26


{'as_of': datetime.date(2026, 9, 25),
 'current_term': 'Fall 2026',
 'current_start': datetime.date(2026, 9, 2),
 'current_end': datetime.date(2026, 12, 15),
 'elapsed_days': 23,
 'total_days': 104,
 'fraction_elapsed': 0.22115384615384615,
 'comparison_term': 'Fall 2025',
 'comparison_window_end': datetime.date(2025, 9, 26)}

## Write the tidy files

Everything downstream reads these, so re-running notebook 00 is how a new data drop propagates.

In [7]:
messages_df.to_csv(CONFIG.data_path("messages.csv"), index=False)
sessions_df.to_csv(CONFIG.data_path("sessions.csv"), index=False)

enrolments_df = load.load_enrolments()
enrolments_df.to_csv(CONFIG.data_path("enrolments.csv"), index=False)
print(f"{len(enrolments_df):,} active enrolments across {enrolments_df['course'].nunique()} courses")

metrics.update("dataset", {
    **provenance,
    **summary,
    **completeness,
    "gap_minutes": CONFIG.session_gap_minutes,
    "timezone": CONFIG.display_timezone,
    "enrolments": len(enrolments_df),
    "courses": sorted(messages_df["course"].unique().tolist()),
    "surfaces": sorted(messages_df["surface"].unique().tolist()),
})
print("wrote", CONFIG.metrics_path)

124 active enrolments across 4 courses
wrote /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out/metrics.json
